cbl vs panel vs UCL vs JAX vs Neural SRP grid. Validation vs truth IGS ephemerides

In [1]:
import base
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import pandas as pd
from collections import namedtuple
from tqdm import tqdm

import integrator
import physics
import utils
import map_srp, force_modeling, neural_srp
import cannonball_srp
import nn


In [2]:
MASS = 1633.0
PANEL_AREA = 25.45
PANEL_REFLECTIVITY = 0.28
PANEL_SPECULARITY = 1.0
CBL_AREA = 26.45
CBL_CR = 1.2
STEP_SIZE = 30.0
GRAVITY_DEGREE = 24
PROPAGATION_HOURS = 12.0

In [3]:
path_to_grids = base.drive_path + "srp_grids/"

jax_grid_x_nb = map_srp.read_grd_file(path_to_grids + 'JAX/gps2f_busonly/no_brdf/gps2f_nb_force_rot_X.grd')
jax_grid_y_nb = map_srp.read_grd_file(path_to_grids + 'JAX/gps2f_busonly/no_brdf/gps2f_nb_force_rot_Y.grd')
jax_grid_z_nb = map_srp.read_grd_file(path_to_grids + 'JAX/gps2f_busonly/no_brdf/gps2f_nb_force_rot_Z.grd')

ucl_grid_x = map_srp.read_grd_file(path_to_grids + 'UCL/gps2f_busonly/gpsiif_x.grd')
ucl_grid_y = map_srp.read_grd_file(path_to_grids + 'UCL/gps2f_busonly/gpsiif_y.grd')
ucl_grid_z = map_srp.read_grd_file(path_to_grids + 'UCL/gps2f_busonly/gpsiif_z.grd')

nn_weights_path = base.drive_path + "nn_weights/gps2f/"
nn_params = nn.load_nn_parameters(nn_weights_path + 'parameters.bin')
nn_force_stats = nn.load_nn_parameters(nn_weights_path + 'force_stats.bin')

In [4]:
# Single trajectory file - change this path as needed
CSV_FILE = Path(base.drive_path + "trajectories/gps08.csv")
print(f"Trajectory file: {CSV_FILE.name}")

Trajectory file: gps08.csv


In [5]:
# SRP methods - using module functions where possible
def compute_srp_cannonball(pos, sun):
    shadow = physics.eclipse_model(pos, sun)
    acc = cannonball_srp.compute_srp_cbl(pos, sun, area=CBL_AREA, mass=MASS, cr=CBL_CR)
    return jnp.where(shadow == 0, jnp.zeros(3), acc * shadow)

def compute_srp_panel(pos, sun):
    return cannonball_srp.compute_srp_panel(pos, sun, area=PANEL_AREA, mass=MASS,
                                            reflectivity=PANEL_REFLECTIVITY, specularity=PANEL_SPECULARITY)

def compute_srp_grid(pos, sun, gx, gy, gz):
    return map_srp.compute_srp_grd(pos, sun, gx, gy, gz) + compute_srp_panel(pos, sun)

compute_srp_ucl = lambda pos, sun: compute_srp_grid(pos, sun, ucl_grid_x, ucl_grid_y, ucl_grid_z)
compute_srp_jax = lambda pos, sun: compute_srp_grid(pos, sun, jax_grid_x_nb, jax_grid_y_nb, jax_grid_z_nb)
compute_srp_neural = lambda pos, sun: neural_srp.neural_srp_model(pos, sun, nn_params, nn_force_stats)

In [6]:
def make_force_model(srp_fn):
    def model(state, time, parameters, *args):
        pos, vel = state[0], state[1]
        moon_pos, sun_pos, ref_times = args[0]
        moon = utils.sample_moonsun_cubic(time, moon_pos, ref_times)
        sun = utils.sample_moonsun_cubic(time, sun_pos, ref_times)
        gravity = physics.compute_gravity(pos) + physics.compute_gravity_HOT(pos, time, GRAVITY_DEGREE)
        third_body = physics.compute_third_body_acceleration(pos, moon, physics.moon_mass) + \
                     physics.compute_third_body_acceleration(pos, sun, physics.sun_mass)
        return jnp.array([vel, gravity + third_body + srp_fn(pos, sun)])
    return model

In [7]:
def propagate_and_compare(trajectory, force_model, hours):
    conditions = (trajectory.moon, trajectory.sun, trajectory.time)
    
    # Slice to requested duration
    end_time = trajectory.time[0] + hours * 3600.0
    end_idx = int(np.searchsorted(np.array(trajectory.time), end_time, side='right'))
    
    ts, positions = integrator.propagate(
        utils.Trajectory(trajectory.time[:end_idx], trajectory.position[:end_idx],
                        trajectory.velocity[:end_idx], trajectory.moon[:end_idx], trajectory.sun[:end_idx]),
        force_model, STEP_SIZE, None, conditions)
    
    # Match propagated positions to reference epochs
    prop_times = np.array(ts) / 86400
    ref_times = np.array(trajectory.time[:end_idx]) / 86400
    ref_pos = np.array(trajectory.position[:end_idx])
    prop_pos = np.array(positions)
    
    distances = []
    for rt, rp in zip(ref_times, ref_pos):
        idx = np.argmin(np.abs(prop_times - rt))
        if abs(rt - prop_times[idx]) <= 0.01:
            distances.append(float(np.linalg.norm(rp - prop_pos[idx])))
    
    distances = np.array(distances[:-1])  # drop last (identical to IC)
    return distances

In [8]:
SRP_METHODS = {
    'cannonball': compute_srp_cannonball,
    'panel': compute_srp_panel,
    'ucl': compute_srp_ucl,
    'jax': compute_srp_jax,
    'neural': compute_srp_neural,
}

}

SyntaxError: unmatched '}' (2190792657.py, line 9)

In [ ]:
sc_name = CSV_FILE.stem
full_traj = utils.load_trajectory(str(CSV_FILE), limit=None)
print(f"Processing {sc_name} ({PROPAGATION_HOURS}h)...")

results = {}
for name, srp_fn in SRP_METHODS.items():
    model = make_force_model(srp_fn)
    distances = propagate_and_compare(full_traj, model, PROPAGATION_HOURS)
    results[name] = {
        'errors': distances,
        'mean': float(np.mean(distances)),
        'rms': float(np.sqrt(np.mean(distances**2))),
        'max': float(np.max(distances)),
    }
    print(f"  {name:12s}  RMS={results[name]['rms']:.3f}m  Mean={results[name]['mean']:.3f}m")

In [ ]:
print(f"\nBENCHMARK RESULTS FOR {sc_name}")
print(f"{'Method':>12s}  {'Mean (m)':>10s}  {'RMS (m)':>10s}  {'Max (m)':>10s}")
print("-" * 50)
for name in SRP_METHODS:
    r = results[name]
    print(f"{name:>12s}  {r['mean']:10.3f}  {r['rms']:10.3f}  {r['max']:10.3f}")